# Pause & Coherence Pipeline — AVH Dataset

This notebook runs the [`pause_coherence`](https://github.com/chenfeng1234567/pause_coherence) pipeline on the **AVH (Auditory Verbal Hallucinations)** dataset.

It performs Leave-One-Subject-Out (LOSO) cross-validation with 8 feature-fusion approaches:
1. `time_stats_only` — 6 statistical pause features
2. `time_featuredict_only` — 764 tsfresh pause features
3. `coherence_nltk_only` — 764 coherence features (NLTK sentence split)
4. `coherence_whisper_only` — 764 coherence features (Whisper word split)
5. `early_fusion_stats` — time stats + coherence
6. `early_fusion_featuredict` — tsfresh features + coherence
7. `late_fusion_stats` — averaged predictions from stats + coherence
8. `late_fusion_featuredict` — averaged predictions from tsfresh + coherence

**Target variable:** `numscores` (AVH symptom severity)


## 1. Imports

In [ ]:
import sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Add repo root to path so pause_coherence package is importable
sys.path.insert(0, str(Path('').resolve()))

from pause_coherence import (
    AVH_CONFIG,
    COHERENCE_KEYS,
    STATS_FEATURES,
    BaseCoherenceModel,
    ModelResults,
)
from pause_coherence.utils import load_feature_dict, preprocess_dataframe, get_subject_ids

print('pause_coherence package imported successfully.')
print(f'AVH target column : {AVH_CONFIG.target_column}')
print(f'AVH AUC threshold : {AVH_CONFIG.auc_threshold}')


## 2. Path Configuration

Set the paths to the pre-computed AVH data files below.

| Variable | Description |
|---|---|
| `NLTK_PATH` | `.pkl` featureDict — coherence features, NLTK sentence split |
| `WHISPER_PATH` | `.pkl` featureDict — coherence features, WhisperX word split |
| `TIME_STATS_PATH` | `.csv` — 6 statistical pause features (`max`, `min`, `mean_y`, `median_y`, `length`, `pause_proportion`) |
| `TIME_FEAT_PATH` | `.csv` — 764 tsfresh pause features |
| `BASELINE_PATH` | `.jsonl` — participant baseline with `numscores` column |


In [ ]:
# ── Paths to pre-computed AVH data files ─────────────────────────────────────
# Update these paths to point to your data on the server.
# Default values match the filenames referenced in the pause_coherence AVH config.

AVH_BASE_DIR = '/edata/changye/restore/avh-data'  # adjust as needed

NLTK_PATH    = f'{AVH_BASE_DIR}/tald_featureDict_nltk_split_whisperx_310.pkl'
WHISPER_PATH = f'{AVH_BASE_DIR}/tald_featureDict_nltk_split_whisperx_310.pkl'  # update if separate whisper pkl exists
TIME_STATS_PATH  = f'{AVH_BASE_DIR}/tald_pause_time_stats.csv'
TIME_FEAT_PATH   = f'{AVH_BASE_DIR}/tald_pause_tsfresh_features.csv'  # 764 tsfresh features; generate with pause_coherence.preprocessing if needed
BASELINE_PATH    = f'{AVH_BASE_DIR}/avh_tald_310.jsonl'

# ── Model settings ────────────────────────────────────────────────────────────
DATASET      = 'avh'
MODEL_TYPE   = 'svr'    # options: 'svr', 'ridge', 'pls', 'linear_svr', 'elastic_net', 'random_forest'
N_FEATURES   = 200      # top-k features via SelectKBest; set to None to use all
AUC_QUANTILE = 0.8      # quantile used to binarise target for AUC calculation

# Coherence key to use for the single-key run (Cell 5).
# For AVH / Tang the SimCSE cumulative centroid tends to perform best.
COHERENCE_KEY = 'sentCoherenceSimCSECumulativeCentroid'

CONFIG = AVH_CONFIG
print(f'Dataset          : {DATASET}')
print(f'Model type       : {MODEL_TYPE}')
print(f'Number of features: {N_FEATURES}')
print(f'AUC quantile     : {AUC_QUANTILE}')
print(f'Coherence key    : {COHERENCE_KEY}')


## 3. Load Data

In [ ]:
from pause_coherence.utils import split_file_name


def load_baseline_avh(path):
    """Load AVH baseline JSONL and keep rows with a valid numscores value."""
    p = Path(path)
    if p.suffix == '.jsonl':
        df = pd.read_json(path, lines=True)
    elif p.suffix == '.pkl':
        df = pd.read_pickle(path)
    else:
        df = pd.read_csv(path)
    return df.dropna(subset=[CONFIG.target_column])


def load_coherence_avh(path, key):
    """Load coherence featureDict and extract the given key."""
    return load_feature_dict(path, key)


def load_time_stats_avh(path):
    """Load 6-feature pause time stats CSV."""
    df = pd.read_csv(path)
    # Normalise column names used in older exports
    if 'mean' in df.columns and 'mean_y' not in df.columns:
        df = df.rename(columns={'mean': 'mean_y', 'median': 'median_y'})
    return df


def load_time_feat_avh(path):
    """Load 764-feature tsfresh pause features CSV."""
    return pd.read_csv(path)


# ── Load all sources ─────────────────────────────────────────────────────────
tald         = load_baseline_avh(BASELINE_PATH)
sent_nltk    = load_coherence_avh(NLTK_PATH, COHERENCE_KEY)
sent_whisper = load_coherence_avh(WHISPER_PATH, COHERENCE_KEY)
time_stats   = load_time_stats_avh(TIME_STATS_PATH)
time_feat    = load_time_feat_avh(TIME_FEAT_PATH)

print(f'Baseline         : {tald.shape}')
print(f'NLTK coherence   : {sent_nltk.shape}')
print(f'Whisper coherence: {sent_whisper.shape}')
print(f'Time stats       : {time_stats.shape}')
print(f'Time featuredict : {time_feat.shape}')

# ── Compute AUC threshold from data ──────────────────────────────────────────
AUC_THRESHOLD = np.quantile(tald[CONFIG.target_column].values, AUC_QUANTILE)
print(f'\nAUC threshold (top {100 - AUC_QUANTILE*100:.0f}%): {AUC_THRESHOLD:.3f}')


## 4. Align on Common Files

In [ ]:
files_baseline  = set(tald['file'].tolist())
files_nltk      = set(sent_nltk['file'].tolist())
files_whisper   = set(sent_whisper['file'].tolist())
files_stats     = set(time_stats['file'].tolist())
files_feat      = set(time_feat['file'].tolist())

common_files = files_baseline & files_nltk & files_whisper & files_stats & files_feat
print(f'Common files across all sources: {len(common_files)}')

tald         = tald[tald['file'].isin(common_files)].reset_index(drop=True)
sent_nltk    = sent_nltk[sent_nltk['file'].isin(common_files)].reset_index(drop=True)
sent_whisper = sent_whisper[sent_whisper['file'].isin(common_files)].reset_index(drop=True)
time_stats   = time_stats[time_stats['file'].isin(common_files)].reset_index(drop=True)
time_feat    = time_feat[time_feat['file'].isin(common_files)].reset_index(drop=True)

n = len(tald)
print(f'\nFiltered shapes (all should be {n}):')
print(f'  Baseline         : {tald.shape}')
print(f'  NLTK coherence   : {sent_nltk.shape}')
print(f'  Whisper coherence: {sent_whisper.shape}')
print(f'  Time stats       : {time_stats.shape}')
print(f'  Time featuredict : {time_feat.shape}')

assert len(sent_nltk) == n
assert len(sent_whisper) == n
assert len(time_stats) == n
assert len(time_feat) == n
print('\nAll sources aligned successfully.')


## 5. Merge Data Sources

In [ ]:
merged_nltk = (
    sent_nltk
    .merge(tald, on='file', how='inner')
    .merge(time_stats, on='file', how='inner')
    .pipe(preprocess_dataframe)
)
print(f'merged_nltk          : {merged_nltk.shape}')

merged_whisper = (
    sent_whisper
    .merge(tald, on='file', how='inner')
    .merge(time_stats, on='file', how='inner')
    .pipe(preprocess_dataframe)
)
print(f'merged_whisper       : {merged_whisper.shape}')

merged_time_feat = (
    time_feat
    .merge(tald, on='file', how='inner')
    .pipe(preprocess_dataframe)
)
print(f'merged_time_feat     : {merged_time_feat.shape}')

merged_nltk_timefeat = (
    sent_nltk
    .merge(tald, on='file', how='inner')
    .merge(time_feat, on='file', how='inner')
    .pipe(preprocess_dataframe)
)
print(f'merged_nltk_timefeat : {merged_nltk_timefeat.shape}')

# time_feat aligned to the same files as merged_nltk_timefeat (needed for late fusion)
time_feat_aligned = (
    time_feat[time_feat['file'].isin(set(merged_nltk_timefeat['file']))]
    .pipe(preprocess_dataframe)
)
print(f'time_feat_aligned    : {time_feat_aligned.shape}')


## 6. Run All 8 Model Approaches

Uses `BaseCoherenceModel` with Leave-One-Subject-Out cross-validation.
Subject IDs are extracted from AVH file names (e.g. `u00000509@avh-20180723-1` → `509`).


In [ ]:
model = BaseCoherenceModel(
    auc_threshold=AUC_THRESHOLD,
    dataset_name=DATASET,
    model_type=MODEL_TYPE,
    n_features=N_FEATURES,
)

results = {}

print('1. NLTK Coherence Only...')
r = model.train_coherence_only(merged_nltk, CONFIG.target_column, split_type='nltk')
results['1_NLTK_Only'] = r
print(f'   Spearman={r.spearman_corr:.4f}  MAE={r.mae:.4f}  AUC={r.auc:.4f}  AUPRC={r.auprc:.4f}')

print('2. Whisper Coherence Only...')
r = model.train_coherence_only(merged_whisper, CONFIG.target_column, split_type='whisper')
results['2_Whisper_Only'] = r
print(f'   Spearman={r.spearman_corr:.4f}  MAE={r.mae:.4f}  AUC={r.auc:.4f}  AUPRC={r.auprc:.4f}')

print('3. Time Stats Only...')
r = model.train_time_stats_only(merged_nltk, CONFIG.target_column)
results['3_TimeStats_Only'] = r
print(f'   Spearman={r.spearman_corr:.4f}  MAE={r.mae:.4f}  AUC={r.auc:.4f}  AUPRC={r.auprc:.4f}')

print('4. Time FeatureDict Only...')
r = model.train_time_featuredict_only(merged_time_feat, CONFIG.target_column)
results['4_TimeFeat_Only'] = r
print(f'   Spearman={r.spearman_corr:.4f}  MAE={r.mae:.4f}  AUC={r.auc:.4f}  AUPRC={r.auprc:.4f}')

print('5. Early Fusion (NLTK + Stats)...')
r = model.train_early_fusion_stats(merged_nltk, CONFIG.target_column)
results['5_EarlyFusion_Stats'] = r
print(f'   Spearman={r.spearman_corr:.4f}  MAE={r.mae:.4f}  AUC={r.auc:.4f}  AUPRC={r.auprc:.4f}')

print('6. Early Fusion (NLTK + TimeFeat)...')
r = model.train_early_fusion_featuredict(merged_nltk_timefeat, time_feat_aligned, CONFIG.target_column)
results['6_EarlyFusion_Feat'] = r
print(f'   Spearman={r.spearman_corr:.4f}  MAE={r.mae:.4f}  AUC={r.auc:.4f}  AUPRC={r.auprc:.4f}')

print('7. Late Fusion (NLTK + Stats)...')
r = model.train_late_fusion_stats(merged_nltk, CONFIG.target_column)
results['7_LateFusion_Stats'] = r
print(f'   Spearman={r.spearman_corr:.4f}  MAE={r.mae:.4f}  AUC={r.auc:.4f}  AUPRC={r.auprc:.4f}')

print('8. Late Fusion (NLTK + TimeFeat)...')
r = model.train_late_fusion_featuredict(merged_nltk_timefeat, time_feat_aligned, CONFIG.target_column)
results['8_LateFusion_Feat'] = r
print(f'   Spearman={r.spearman_corr:.4f}  MAE={r.mae:.4f}  AUC={r.auc:.4f}  AUPRC={r.auprc:.4f}')

print('\nAll 8 model approaches complete.')


## 7. Results Summary

In [ ]:
summary = pd.DataFrame([
    {
        'Model': name.replace('_', ' '),
        'MAE': r.mae,
        'Spearman': r.spearman_corr,
        'P-Value': r.p_value,
        'AUC': r.auc,
        'AUPRC': r.auprc,
    }
    for name, r in results.items()
]).sort_values('Spearman', ascending=False).reset_index(drop=True)

print('=' * 80)
print(f'RESULTS SUMMARY — AVH Dataset')
print(f'Target: {CONFIG.target_column}  |  Coherence key: {COHERENCE_KEY}')
print(f'Model: {MODEL_TYPE}  |  n_features={N_FEATURES}  |  AUC threshold={AUC_THRESHOLD:.3f}')
print('=' * 80)
print(summary.round(4).to_string(index=False))
print('=' * 80)


## 8. AUPRC Quantile Sensitivity Analysis

Re-computes AUC and AUPRC at multiple binarisation thresholds to check how robust the results are.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

QUANTILES = [0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 0.95]
q_rows = []

for approach, r in results.items():
    y_true = r.true_values
    y_pred = r.predictions
    for q in QUANTILES:
        threshold = np.quantile(y_true, q)
        y_bin = (y_true >= threshold).astype(int)
        if len(np.unique(y_bin)) < 2:
            auc_val = auprc_val = np.nan
        else:
            auc_val  = roc_auc_score(y_bin, y_pred)
            auprc_val = average_precision_score(y_bin, y_pred)
        q_rows.append({
            'approach': approach,
            'quantile': q,
            'threshold': threshold,
            'auc': auc_val,
            'auprc': auprc_val,
            'n_positive': int(np.sum(y_bin)),
            'n_total': len(y_true),
        })

q_df = pd.DataFrame(q_rows)
pivot = q_df.pivot(index='approach', columns='quantile', values='auprc').round(3)
print('AUPRC by quantile threshold:')
print(pivot.to_string())


### Quantile Sensitivity Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

PLOT_APPROACHES = {
    '1_NLTK_Only':         'NLTK coherence only',
    '2_Whisper_Only':      'WhisperX coherence only',
    '3_TimeStats_Only':    'Pause stats only',
    '4_TimeFeat_Only':     'Pause tsfresh only',
    '6_EarlyFusion_Feat':  'Early fusion (NLTK + tsfresh)',
    '8_LateFusion_Feat':   'Late fusion (NLTK + tsfresh)',
}

for key, label in PLOT_APPROACHES.items():
    subset = q_df[q_df['approach'] == key]
    y_vals = [subset[subset['quantile'] == q]['auprc'].values[0]
              if len(subset[subset['quantile'] == q]) > 0 else np.nan
              for q in QUANTILES]
    ax.plot(QUANTILES, y_vals, marker='o', label=label)

ax.set_xlabel('Binarisation Quantile')
ax.set_ylabel('AUPRC')
ax.set_xticks(QUANTILES)
ax.set_xticklabels([f'{int(q*100)}%' for q in QUANTILES])
ax.set_xlim(0.45, 1.0)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_title('AVH: AUPRC Quantile Sensitivity', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('avh_quantile_sensitivity.png', dpi=150)
plt.show()
print('Saved avh_quantile_sensitivity.png')


## 9. Full Sweep — All 24 Coherence Configurations

Runs the core 6 model approaches across all 24 coherence embedding × similarity combinations.
This cell can take a while; consider running overnight.


In [ ]:
try:
    from tqdm import tqdm
    _tqdm = tqdm
except ImportError:
    def _tqdm(x, **kw): return x

all_config_results = []

for ckey in _tqdm(COHERENCE_KEYS, desc='Coherence configs'):
    # Load coherence features for this key
    sn = load_coherence_avh(NLTK_PATH, ckey)
    sw = load_coherence_avh(WHISPER_PATH, ckey)

    common = (
        set(tald['file']) & set(sn['file']) & set(sw['file'])
        & set(time_stats['file']) & set(time_feat['file'])
    )
    sn = sn[sn['file'].isin(common)]
    sw = sw[sw['file'].isin(common)]
    tald_f = tald[tald['file'].isin(common)]
    ts_f   = time_stats[time_stats['file'].isin(common)]
    tf_f   = time_feat[time_feat['file'].isin(common)]

    mn   = sn.merge(tald_f, on='file').merge(ts_f, on='file').pipe(preprocess_dataframe)
    mw   = sw.merge(tald_f, on='file').merge(ts_f, on='file').pipe(preprocess_dataframe)
    mtf  = tf_f.merge(tald_f, on='file').pipe(preprocess_dataframe)
    mntf = sn.merge(tald_f, on='file').merge(tf_f, on='file').pipe(preprocess_dataframe)
    tfa  = tf_f[tf_f['file'].isin(set(mntf['file']))].pipe(preprocess_dataframe)

    m = BaseCoherenceModel(
        auc_threshold=AUC_THRESHOLD,
        dataset_name=DATASET,
        model_type=MODEL_TYPE,
        n_features=N_FEATURES,
    )

    rn  = m.train_coherence_only(mn,   CONFIG.target_column, split_type='nltk')
    rw  = m.train_coherence_only(mw,   CONFIG.target_column, split_type='whisper')
    res = m.train_early_fusion_stats(mn, CONFIG.target_column)
    ref = m.train_early_fusion_featuredict(mntf, tfa, CONFIG.target_column)
    rls = m.train_late_fusion_stats(mn,  CONFIG.target_column)
    rlf = m.train_late_fusion_featuredict(mntf, tfa, CONFIG.target_column)

    all_config_results.append({
        'coherence_key':        ckey,
        'nltk_spearman':        rn.spearman_corr,  'nltk_mae':  rn.mae,  'nltk_auprc':  rn.auprc,
        'whisper_spearman':     rw.spearman_corr,  'whisper_mae': rw.mae, 'whisper_auprc': rw.auprc,
        'early_stats_spearman': res.spearman_corr, 'early_stats_mae': res.mae, 'early_stats_auprc': res.auprc,
        'early_feat_spearman':  ref.spearman_corr, 'early_feat_mae': ref.mae, 'early_feat_auprc': ref.auprc,
        'late_stats_spearman':  rls.spearman_corr, 'late_stats_mae': rls.mae, 'late_stats_auprc': rls.auprc,
        'late_feat_spearman':   rlf.spearman_corr, 'late_feat_mae': rlf.mae, 'late_feat_auprc': rlf.auprc,
    })

config_df = pd.DataFrame(all_config_results)
print(f'Completed {len(config_df)} configurations.')
config_df.to_csv('avh_24configs_results.csv', index=False)
print('Saved avh_24configs_results.csv')


### Full-sweep Results Table

In [ ]:
display_cols = [
    'coherence_key',
    'nltk_spearman', 'late_stats_spearman', 'late_feat_spearman',
    'nltk_mae',      'late_stats_mae',      'late_feat_mae',
    'nltk_auprc',    'late_stats_auprc',    'late_feat_auprc',
]
display_df = config_df[display_cols].copy()
display_df.columns = [
    'Coherence Key',
    'NLTK Spearman', 'LF-Stats Spearman', 'LF-Feat Spearman',
    'NLTK MAE',      'LF-Stats MAE',      'LF-Feat MAE',
    'NLTK AUPRC',    'LF-Stats AUPRC',    'LF-Feat AUPRC',
]
print(display_df.round(3).to_string(index=False))


### Full-sweep Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(len(config_df))
w = 0.25

metrics = [
    ('Spearman', 'nltk_spearman', 'late_stats_spearman', 'late_feat_spearman'),
    ('MAE (lower=better)', 'nltk_mae', 'late_stats_mae', 'late_feat_mae'),
    ('AUPRC', 'nltk_auprc', 'late_stats_auprc', 'late_feat_auprc'),
]

for ax, (ylabel, col_nltk, col_ls, col_lf) in zip(axes, metrics):
    ax.bar(x - w, config_df[col_nltk], w, label='NLTK Only',           alpha=0.8)
    ax.bar(x,     config_df[col_ls],   w, label='Late Fusion (Stats)',  alpha=0.8)
    ax.bar(x + w, config_df[col_lf],   w, label='Late Fusion (Feat)',   alpha=0.8)
    ax.set_xlabel('Config #')
    ax.set_ylabel(ylabel)
    ax.set_title(f'AVH: {ylabel}')
    ax.set_xticks(x)
    ax.set_xticklabels([str(i + 1) for i in range(len(config_df))], fontsize=7)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('avh_24configs_barchart.png', dpi=150)
plt.show()
print('Saved avh_24configs_barchart.png')
